In [1]:
from pyhdf.SD import SD, SDC
import numpy as np
import h5py
import os

In [2]:
def read_hdf4_field(file_path):
    """
    Reads Data-Set-2 and coordinate arrays from HDF4 file.
    Returns:
        data, phi, theta, r
    """
    print(f"Reading HDF4 file: {file_path}")
    f = SD(file_path, SDC.READ)

    data  = f.select("Data-Set-2").get()
    phi   = f.select("fakeDim0").get()
    theta = f.select("fakeDim1").get()
    r     = f.select("fakeDim2").get()

    return data, phi, theta, r

In [3]:
def remove_radial_ghost_cells(field):
    """
    Removes first and last radial ghost cells.
    field shape: (phi, theta, Nr)
    """
    return field[..., 1:-1]


def remove_radial_ghost_coords(r):
    """
    Removes first and last radial ghost coordinates.
    """
    return r[1:-1]


def face_to_center_radial(vr_face):
    """
    Convert face-centered radial velocity to cell-centered.
    vr_face shape: (phi, theta, Nr_faces)
    Returns: (phi, theta, Nr_faces - 1)
    """
    return 0.5 * (vr_face[..., 1:] + vr_face[..., :-1])

def face_to_center_theta(field_face):
    """
    Convert theta-face centered field to theta-cell centered.
    field_face shape: (phi, Ntheta_faces, r)
    Returns: (phi, Ntheta_faces-1, r)
    """
    return 0.5 * (field_face[:, 1:, :] + field_face[:, :-1, :])

def face_to_center_theta_coords(theta_face):
    return 0.5 * (theta_face[1:] + theta_face[:-1])

In [4]:
def convert_staggered_spherical(sim_path, output_path):
    """
    Reads vr, rho, p HDF4 files,
    collocates onto same spherical mesh,
    and writes clean HDF5 file.
    """

    # ---- Load files ----
    vr, phi_vr, theta_vr, r_faces = read_hdf4_field(
        os.path.join(sim_path, "vr002.hdf")
    )
    rho, phi_rho, theta_rho, r_center = read_hdf4_field(
        os.path.join(sim_path, "rho002.hdf")
    )
    p, phi_p, theta_p, r_p = read_hdf4_field(
        os.path.join(sim_path, "p002.hdf")
    )

    # ==========================================================
    # 1) Remove radial ghost cells (rho, p are cell-centered)
    # ==========================================================
    rho = remove_radial_ghost_cells(rho)
    p   = remove_radial_ghost_cells(p)
    r_center = remove_radial_ghost_coords(r_center)

    # ==========================================================
    # 2) Interpolate vr from radial faces → radial centers
    # ==========================================================
    vr = face_to_center_radial(vr)

    # ==========================================================
    # 3) Remove theta ghost cells (all variables)
    # ==========================================================
    rho = rho[:, 1:-1, :]
    p   = p[:, 1:-1, :]
    vr  = vr[:, 1:-1, :]
    theta_center = theta_rho[1:-1]

    # ==========================================================
    # 5) Final sanity check
    # ==========================================================
    assert vr.shape == rho.shape == p.shape, (
        f"Shape mismatch: vr{vr.shape}, rho{rho.shape}, p{p.shape}"
    )

    print("Final collocated shape:", vr.shape)

    # ==========================================================
    # 6) Write HDF5
    # ==========================================================
    output_path = os.path.join(output_path, "vr_rho_p_r0.h5")

    with h5py.File(output_path, "w") as f:

        # Fields
        f.create_dataset("vr",  data=vr[..., 0],  compression="gzip", compression_opts=4)
        f.create_dataset("rho", data=rho[..., 0], compression="gzip", compression_opts=4)
        f.create_dataset("p",   data=p[..., 0],   compression="gzip", compression_opts=4)

        # Coordinates
        f.create_dataset("phi",   data=phi_rho)
        f.create_dataset("theta", data=theta_center)
        f.create_dataset("r",     data=[r_center[0]])

        # Metadata
        f.attrs["grid_type"] = "spherical"
        f.attrs["collocation"] = "fully cell-centered"
        f.attrs["dimensions"] = "(phi, theta, r)"
        f.attrs["original_staggering"] = (
            "vr: radial-face centered; "
            "rho,p: cell-centered; "
            "theta: ghost cells removed"
        )

    print(f"Converted and fully collocated → {output_path}")

In [6]:
source = "/Users/reza/Career/DMLab/SURROGATE/Data/all-components/test"
dest = "/Users/reza/Career/DMLab/SURROGATE/Data/all-components/boundary-conditions"

for cr in os.listdir(source):
    if cr.startswith("."):
        continue
    for instrument in os.listdir(os.path.join(source, cr)):
        if instrument.startswith("."):
            continue
        if not os.path.exists(os.path.join(dest, cr, instrument)):
            os.makedirs(os.path.join(dest, cr, instrument))
        convert_staggered_spherical(
            sim_path=os.path.join(
                source,
                cr,
                instrument,
            ),
            output_path=os.path.join(
                dest,
                cr,
                instrument,
            ),
        )

Reading HDF4 file: /Users/reza/Career/DMLab/SURROGATE/Data/all-components/test/cr2201/hmi_mast_mas_std_0101/vr002.hdf
Reading HDF4 file: /Users/reza/Career/DMLab/SURROGATE/Data/all-components/test/cr2201/hmi_mast_mas_std_0101/rho002.hdf
Reading HDF4 file: /Users/reza/Career/DMLab/SURROGATE/Data/all-components/test/cr2201/hmi_mast_mas_std_0101/p002.hdf
Final collocated shape: (128, 109, 139)
Converted and fully collocated → /Users/reza/Career/DMLab/SURROGATE/Data/all-components/boundary-conditions/cr2201/hmi_mast_mas_std_0101/vr_rho_p_r0.h5
Reading HDF4 file: /Users/reza/Career/DMLab/SURROGATE/Data/all-components/test/cr2201/hmi_mast_mas_std_0201/vr002.hdf
Reading HDF4 file: /Users/reza/Career/DMLab/SURROGATE/Data/all-components/test/cr2201/hmi_mast_mas_std_0201/rho002.hdf
Reading HDF4 file: /Users/reza/Career/DMLab/SURROGATE/Data/all-components/test/cr2201/hmi_mast_mas_std_0201/p002.hdf
Final collocated shape: (128, 109, 139)
Converted and fully collocated → /Users/reza/Career/DMLab/SU